# Dietary tagging EDA — `food_4macro` + resolved recipes

Coverage and distribution analysis for diabetes, osteoporosis, and restriction tagging.

**Kernel cwd:** `Capstone/` project root (or `scratch/EDA/`). Requires `.env` Postgres credentials.

See also: `docs/dietary_tagging_framework.md`, `scripts/tag_eda.py`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def resolve_capstone_root() -> Path:
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "scripts" / "db.py").is_file():
            return candidate
    raise FileNotFoundError("Run with kernel cwd = Capstone/ or scratch/EDA/")


ROOT = resolve_capstone_root()
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from db import connect, load_dotenv
from tag_dimensions import NUTRIENT_DIMENSIONS, TAG_NUTRIENT_IDS

load_dotenv()
print("Capstone root:", ROOT)

## 1. Nutrient coverage on `food_4macro`

In [ ]:
COVERAGE_SQL = """
SELECT n.id, n.name, n.unit_name,
       COUNT(DISTINCT fn.fdc_id) AS n_foods,
       ROUND(100.0 * COUNT(DISTINCT fn.fdc_id) /
         NULLIF((SELECT COUNT(*) FROM usda.food_4macro), 0), 2) AS pct_of_catalog
FROM usda.nutrient n
LEFT JOIN usda.food_nutrient fn
  ON fn.nutrient_id = n.id
 AND fn.fdc_id IN (SELECT fdc_id FROM usda.food_4macro)
 AND fn.amount IS NOT NULL
WHERE n.id = ANY(%s)
GROUP BY n.id, n.name, n.unit_name
ORDER BY n.id
"""

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM usda.food_4macro")
        n_catalog = int(cur.fetchone()[0])
        cur.execute(COVERAGE_SQL, (list(TAG_NUTRIENT_IDS),))
        cov = pd.DataFrame(
            cur.fetchall(),
            columns=["id", "name", "unit_name", "n_foods", "pct_of_catalog"],
        )

print(f"food_4macro: {n_catalog:,} foods")
display(cov)

## 2. Nutrient coverage on resolved recipe lines

In [ ]:
RESOLVED_SQL = """
SELECT fn.nutrient_id, n.name,
       COUNT(*) AS n_lines_with_value,
       COUNT(DISTINCT rr.recipe_id) AS n_recipes
FROM recipe.resolved_recipes rr
JOIN usda.food_nutrient fn ON fn.fdc_id = rr.fdc_id AND fn.amount IS NOT NULL
JOIN usda.nutrient n ON n.id = fn.nutrient_id
WHERE rr.fdc_id IS NOT NULL AND fn.nutrient_id = ANY(%s)
GROUP BY fn.nutrient_id, n.name
ORDER BY fn.nutrient_id
"""

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*), COUNT(DISTINCT recipe_id) FROM recipe.resolved_recipes")
        n_lines, n_recipes = cur.fetchone()
        cur.execute(RESOLVED_SQL, (list(TAG_NUTRIENT_IDS),))
        resolved_cov = pd.DataFrame(
            cur.fetchall(),
            columns=["nutrient_id", "name", "n_lines_with_value", "n_recipes"],
        )

print(f"resolved_recipes: {n_lines:,} lines, {n_recipes:,} recipes")
display(resolved_cov)

## 3. Distribution histograms (per-100g on catalog sample)

In [ ]:
SAMPLE_SQL = """
SELECT fn.nutrient_id, fn.amount
FROM usda.food_nutrient fn
WHERE fn.fdc_id IN (SELECT fdc_id FROM usda.food_4macro TABLESAMPLE BERNOULLI(1))
  AND fn.nutrient_id = ANY(%s)
  AND fn.amount IS NOT NULL
"""

plot_dims = [d for d in NUTRIENT_DIMENSIONS if d.slug in ("sodium", "fiber", "protein", "calcium")]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

with connect() as conn:
    with conn.cursor() as cur:
        for ax, dim in zip(axes, plot_dims):
            cur.execute(
                """
                SELECT fn.amount FROM usda.food_nutrient fn
                WHERE fn.fdc_id IN (SELECT fdc_id FROM usda.food_4macro LIMIT 5000)
                  AND fn.nutrient_id = %s AND fn.amount IS NOT NULL
                """,
                (dim.nutrient_id,),
            )
            vals = [float(r[0]) for r in cur.fetchall()]
            ax.hist(vals, bins=40, color="steelblue", edgecolor="white")
            ax.set_title(f"{dim.slug} ({dim.unit} per 100g)")
            ax.set_xlabel(dim.unit)

plt.tight_layout()
plt.show()

## 4. Restriction / allergen source coverage

In [ ]:
BRANDED_SQL = """
SELECT
  COUNT(*) AS n_food_4macro,
  COUNT(bf.fdc_id) AS n_with_branded_row,
  COUNT(bf.ingredients) FILTER (
    WHERE bf.ingredients IS NOT NULL AND bf.ingredients <> ''
  ) AS n_with_ingredients_text
FROM usda.food_4macro f
LEFT JOIN usda.branded_food bf ON bf.fdc_id = f.fdc_id
"""

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(BRANDED_SQL)
        branded = pd.DataFrame(
            [cur.fetchone()],
            columns=["n_food_4macro", "n_with_branded_row", "n_with_ingredients_text"],
        )

display(branded)
branded["pct_branded"] = (
    100.0 * branded["n_with_branded_row"] / branded["n_food_4macro"]
).round(2)
branded["pct_ingredients_text"] = (
    100.0 * branded["n_with_ingredients_text"] / branded["n_food_4macro"]
).round(2)
display(branded)

## 5. Tagging feasibility summary

Post results to the GitHub issue using the template in `docs/dietary_tagging_eda_report.md`.